# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/atulpatel-net/FlyRank_ML_In/blob/main/work/notebooks/w06_validation_audit.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

### Finding 1 — The Content Results Curve

The paper reports that content health peaks around the 61–90 day age range, with an average Health Score of 37.2. It then falls to 29.6 for content aged 271–365 days, while the 365+ group averages 35.5.

**Methodology question:**

The outcome here is the reported Health Score, which combines impressions, position, CTR, and scroll depth. I would ask whether the age groups are sufficiently comparable in factors such as content type, client mix, and historical visibility. Older and newer pages may differ systematically, so the observed relationship between content age and Health Score does not by itself establish that age causes the change in performance.

A stronger validation design could compare more closely matched age cohorts or control for important differences between groups. I would therefore describe this as an **observed age-performance pattern**, rather than a causal effect of content age.

### Finding 2 — The Freshness Multiplier

The paper reports that recently refreshed content can show stronger performance than stale content. One reported comparison shows a 5.43:1 growth-to-decline ratio for the 31–90 day freshness window, while a separate 365+ refreshed-versus-stale comparison reports a 52× impression difference.

**Methodology question:**

The outcome is based on observed growth, health, and impression differences between freshness groups. I would ask whether pages selected for refreshing were already different from stale pages before the refresh. For example, stronger pages may have been more likely to receive attention or be selected for improvement.

If refresh decisions were influenced by prior performance, the observed difference could partly reflect selection effects rather than the refresh itself. A stronger validation design would compare refreshed pages with a comparable untreated group or use a carefully defined before-and-after design.

Therefore, the result supports an **observed association between freshness and performance**, but it should not automatically be interpreted as proof that refreshing a page caused the measured improvement.

## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

### Validation question

The Week-5 model used a client-grouped split for evaluation. To audit the validation design, I compare this with a row-level random split using the same dataset, target definition, features, Random Forest configuration, and evaluation metrics.

The random split represents the less restrictive evaluation because observations from the same client can appear in both training and test sets. The grouped split prevents client overlap between the two partitions.

The purpose of this comparison is to measure how much the validation result changes when the evaluation is performed on previously unseen clients.

In [19]:
import duckdb
import pandas as pd
import numpy as np

from google.colab import userdata

# Load Hugging Face token
HF_TOKEN = userdata.get("HF_TOKEN")

# DuckDB connection
con = duckdb.connect()

con.execute(
    f"CREATE OR REPLACE SECRET hf "
    f"(TYPE huggingface, TOKEN '{HF_TOKEN}')"
)

# FlyRank warehouse
REL = "hf://datasets/FlyRank/internship-warehouse"

# Build the Week-5 feature dataset from all monthly partitions
data = con.sql("""
WITH bounds AS (
    SELECT MAX(report_date) AS end_d
    FROM read_parquet(
        'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=*/data_0.parquet'
    )
),

features AS (
    SELECT
        client_hash_id,
        content_hash_id,

        SUM(
            CASE
                WHEN report_date BETWEEN end_d - INTERVAL 59 DAY
                                     AND end_d - INTERVAL 30 DAY
                THEN gsc_impressions
                ELSE 0
            END
        ) AS imp_prev30,

        SUM(
            CASE
                WHEN report_date BETWEEN end_d - INTERVAL 29 DAY
                                     AND end_d
                THEN gsc_impressions
                ELSE 0
            END
        ) AS imp_last30,

        SUM(
            CASE
                WHEN report_date BETWEEN end_d - INTERVAL 59 DAY
                                     AND end_d - INTERVAL 30 DAY
                THEN gsc_clicks
                ELSE 0
            END
        ) AS clk_prev30,

        AVG(
            CASE
                WHEN report_date BETWEEN end_d - INTERVAL 59 DAY
                                     AND end_d - INTERVAL 30 DAY
                THEN gsc_avg_position
            END
        ) AS pos_prev30

    FROM read_parquet(
        'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=*/data_0.parquet'
    ), bounds

    GROUP BY
        client_hash_id,
        content_hash_id
)

SELECT *
FROM features
WHERE imp_prev30 > 0
""").df()

print("Dataset shape:", data.shape)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Dataset shape: (237429, 6)


In [15]:
### Dataset and target
data["is_declining"] = (
    data["imp_last30"] < 0.8 * data["imp_prev30"]
).astype(int)

feature_cols = [
    "imp_prev30",
    "clk_prev30",
    "pos_prev30"
]

X = data[feature_cols]
y = data["is_declining"]
groups = data["client_hash_id"]

print("Features:", feature_cols)
print("Target rows:", len(y))
print("Declining:", y.sum())
print("Non-declining:", (y == 0).sum())
print("Unique clients:", groups.nunique())

Features: ['imp_prev30', 'clk_prev30', 'pos_prev30']
Target rows: 237429
Declining: 169864
Non-declining: 67565
Unique clients: 56


In [16]:
### Before: Random row-level split
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

X_train_random, X_test_random, y_train_random, y_test_random = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=42,
    stratify=y
)

rf_random = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    n_jobs=-1
)

rf_random.fit(X_train_random, y_train_random)

rf_random_pred = rf_random.predict(X_test_random)

print("Random row-level split")
print(f"Accuracy:  {accuracy_score(y_test_random, rf_random_pred):.3f}")
print(f"Precision: {precision_score(y_test_random, rf_random_pred, zero_division=0):.3f}")
print(f"Recall:    {recall_score(y_test_random, rf_random_pred, zero_division=0):.3f}")
print(f"F1:        {f1_score(y_test_random, rf_random_pred, zero_division=0):.3f}")

Random row-level split
Accuracy:  0.669
Precision: 0.742
Recall:    0.825
F1:        0.781


In [17]:
### After: Client-grouped split
from sklearn.model_selection import GroupShuffleSplit

gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.25,
    random_state=42
)

train_idx, test_idx = next(
    gss.split(X, y, groups)
)

X_train_grouped = X.iloc[train_idx]
X_test_grouped = X.iloc[test_idx]

y_train_grouped = y.iloc[train_idx]
y_test_grouped = y.iloc[test_idx]

rf_grouped = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    n_jobs=-1
)

rf_grouped.fit(X_train_grouped, y_train_grouped)

rf_grouped_pred = rf_grouped.predict(X_test_grouped)

grouped_accuracy = accuracy_score(
    y_test_grouped,
    rf_grouped_pred
)

grouped_precision = precision_score(
    y_test_grouped,
    rf_grouped_pred,
    zero_division=0
)

grouped_recall = recall_score(
    y_test_grouped,
    rf_grouped_pred,
    zero_division=0
)

grouped_f1 = f1_score(
    y_test_grouped,
    rf_grouped_pred,
    zero_division=0
)

print("Client-grouped split")
print(f"Accuracy:  {grouped_accuracy:.3f}")
print(f"Precision: {grouped_precision:.3f}")
print(f"Recall:    {grouped_recall:.3f}")
print(f"F1:        {grouped_f1:.3f}")

print("\nClient overlap:")
print(len(set(groups.iloc[train_idx]) & set(groups.iloc[test_idx])))

Client-grouped split
Accuracy:  0.616
Precision: 0.670
Recall:    0.812
F1:        0.735

Client overlap:
0


In [18]:
comparison = pd.DataFrame({
    "Split": [
        "Random row-level",
        "Client-grouped"
    ],
    "Accuracy": [
        accuracy_score(y_test_random, rf_random_pred),
        grouped_accuracy
    ],
    "Precision": [
        precision_score(y_test_random, rf_random_pred, zero_division=0),
        grouped_precision
    ],
    "Recall": [
        recall_score(y_test_random, rf_random_pred, zero_division=0),
        grouped_recall
    ],
    "F1": [
        f1_score(y_test_random, rf_random_pred, zero_division=0),
        grouped_f1
    ]
})

display(comparison.round(3))

,Split,Accuracy,Precision,Recall,F1
0,Random row-level,0.669,0.742,0.825,0.781
1,Client-grouped,0.616,0.670,0.812,0.735


### Interpretation

The random row-level split produced a higher measured F1 score (0.781) than the client-grouped split (0.735), a difference of 0.046.

The random split allows observations from the same client to appear in both the training and test sets. The grouped split prevents this overlap, with zero clients shared between the two partitions.

The lower grouped result provides a more conservative measurement of performance when the intended setting involves previously unseen clients. The difference between the two evaluations shows that validation design affects the measured performance of this model.

Therefore, the client-grouped result (F1 = 0.735) is used as the more appropriate reference for this audit. This result indicates observed predictive performance under the evaluated grouped split; it does not establish performance for all clients or guarantee future prediction quality.

## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

### Leakage question

The target `is_declining` is defined using `imp_last30` compared with `imp_prev30`. The model features should therefore contain information available before the target window and should not directly use the target period.

I audit each feature based on its time window and its relationship to the target definition. I also check that identifier columns are used only for grouping and are not included as model features.

In [20]:
leakage_audit = pd.DataFrame({
    "Column": [
        "imp_prev30",
        "clk_prev30",
        "pos_prev30",
        "imp_last30",
        "client_hash_id",
        "content_hash_id"
    ],
    "Role": [
        "Model feature",
        "Model feature",
        "Model feature",
        "Target construction",
        "Grouping / split identifier",
        "Content identifier"
    ],
    "Time relationship": [
        "Previous 30-day window",
        "Previous 30-day window",
        "Previous 30-day window",
        "Last 30-day target window",
        "Identifier",
        "Identifier"
    ],
    "Leakage assessment": [
        "No",
        "No",
        "No",
        "Not used as a feature",
        "Not used as a feature",
        "Not used as a feature"
    ],
    "Reason": [
        "Measured before the target window.",
        "Measured before the target window.",
        "Measured before the target window.",
        "Used to define the target, so using it as a feature would directly leak the outcome.",
        "Used to prevent client overlap between train and test.",
        "Used to identify content, not as a predictive feature."
    ]
})

display(leakage_audit)

,Column,Role,Time relationship,Leakage assessment,Reason
0,imp_prev30,Model feature,Previous 30-day window,No,Measured before the target window.
1,clk_prev30,Model feature,Previous 30-day window,No,Measured before the target window.
2,pos_prev30,Model feature,Previous 30-day window,No,Measured before the target window.
3,imp_last30,Target construction,Last 30-day target window,Not used as a feature,"Used to define the target, so using it as a fe..."
4,client_hash_id,Grouping / split identifier,Identifier,Not used as a feature,Used to prevent client overlap between train a...
5,content_hash_id,Content identifier,Identifier,Not used as a feature,"Used to identify content, not as a predictive ..."


In [21]:
print("Features used by model:")
print(feature_cols)

print("\nTarget:")
print("is_declining")

print("\nPotential target column included in features:")
print("imp_last30" in feature_cols)

Features used by model:
['imp_prev30', 'clk_prev30', 'pos_prev30']

Target:
is_declining

Potential target column included in features:
False


### Leakage conclusion

No direct target leakage was identified in the three model features used in this audit. `imp_prev30`, `clk_prev30`, and `pos_prev30` are calculated from the 30-day period preceding the target window.

`imp_last30` is used to construct `is_declining` but is not included as a model feature. Including it would leak information from the target period.

The client and content identifiers are also excluded from the feature set. `client_hash_id` is used only for grouped validation, which prevents the same client from appearing in both training and test partitions.

Based on this audit, the evaluated feature set has a clear temporal separation from the target and no direct feature-level leakage was identified.

## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

### Why the claims need review

The Week-5 evaluation showed strong measured performance for the Random Forest under the client-grouped split, with an F1 score of 0.735, precision of 0.670, and recall of 0.814.

However, the Week-6 audit shows that the measured F1 changes from 0.781 under a random row-level split to 0.735 under a client-grouped split. This demonstrates that the validation design affects the measured result.

The claims below are therefore rewritten to describe what was observed and measured, without extending the evidence to causal, universal, or production-level conclusions.

### Week-5 claim → Week-6 evidence-safe claim

**Original claim:**

> The Random Forest provides a significant improvement over the Week-4 rule-based baseline and can effectively identify declining content.

**Rewritten claim:**

> Under the evaluated client-grouped split, the Random Forest achieved a measured F1 score of 0.735, compared with 0.018 for the Week-4 rule-based baseline. The result shows stronger observed classification performance for the Random Forest on this evaluation set and supports its use as directional decision-support for identifying content matching the defined decline label.

**Why this is safer:**

The evidence supports a measured performance comparison on the evaluated dataset and split. It does not establish that the model will perform equally well for all clients, that it will generalize to future data, or that the model explains or causes content performance declines.

### Validation claim

**Overstated version:**

> The model achieves 78.1% F1 and is reliable for predicting content decline.

**Evidence-safe version:**

> The model achieved a measured F1 score of 0.781 under a random row-level split and 0.735 under a client-grouped split. Because the grouped split prevents client overlap between training and test data, the grouped result is used as the more conservative reference for this audit. These results indicate observed performance under the evaluated validation design rather than guaranteed performance on future or unseen data.

### Claim language used in this audit

The analysis uses:

- **Observed** when describing patterns found in the evaluated data.
- **Measured** when reporting model metrics or calculated differences.
- **Directional** when describing the potential usefulness of a model or finding without implying certainty.
- **Decision-support** when describing how the model could assist analysis or prioritization.

The analysis avoids claims of causation, universal generalization, guaranteed performance, or production readiness because those conclusions are not established by the current validation design.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.